# Multiverse Hybrid v3.0 — Stage 0 PRICE Bulk 2000

**認可済みPRICE-only一括復元専用です。**

- 2000Rの保存済みSHA-bound rawだけをオフライン再解析
- PRICE / 市場存在 / active car情報だけ出力
- Settlement / 払戻 / RESULT / EV / ROI は実行しません
- ECON_HOLDOUT1000 は SEALED のまま
- 未知構造はWeb補完せずFail-Closed記録
- 完了後にPost-Bulk Quality Reportを自動生成

iPhoneでは **ランタイム → すべてのセルを実行** だけで構いません。


In [ ]:
!pip -q install lxml

from google.colab import drive, files
from pathlib import Path
import subprocess, shutil, json, os

drive.mount('/content/drive')

REPO=Path('/content/multiverse-research-stage0-price')
if REPO.exists(): shutil.rmtree(REPO)
subprocess.check_call(['git','clone','--depth','1','https://github.com/fufufu1116/multiverse-research.git',str(REPO)])

EXPECTED={
 'v3/historical_all_market/stage0_price_bulk_runner_v1.py':'01345100dba64955a080a671f5da3ceba1bc23c0',
 'v3/historical_all_market/kdreams_price_catalog_recovery_v1.py':'f94a08a3ea7c0a4f110dc0df82433eecc25b0cf8',
 'v3/historical_all_market/governance/INDEPENDENT_GEMINI_STAGE0_FINAL_APPROVE_RECEIPT_v1.json':'8643684cf7bf0165968ae667e17e546936a3611d',
}
for rel,exp in EXPECTED.items():
    obs=subprocess.check_output(['git','-C',str(REPO),'hash-object',rel],text=True).strip()
    if obs!=exp: raise RuntimeError(f'FAIL-CLOSED Git blob mismatch {rel}: {obs} != {exp}')
print('✅ EXACT CODE / APPROVAL BINDINGS PASS')

approve=json.loads((REPO/'v3/historical_all_market/governance/INDEPENDENT_GEMINI_STAGE0_FINAL_APPROVE_RECEIPT_v1.json').read_text())
if approve.get('verdict')!='APPROVE': raise RuntimeError('FAIL-CLOSED Gemini verdict')
if approve.get('authorization',{}).get('price_only_stage0_bulk_2000')!='AUTHORIZED': raise RuntimeError('FAIL-CLOSED PRICE bulk not authorized')
if approve.get('authorization',{}).get('settlement_bulk_now')!='PROHIBITED': raise RuntimeError('FAIL-CLOSED settlement prohibition missing')

runner=REPO/'v3/historical_all_market/stage0_price_bulk_runner_v1.py'
subprocess.check_call([
    'python',str(runner),
    '--mydrive','/content/drive/MyDrive',
    '--repo-root',str(REPO),
    '--overwrite'
])

OUT=Path('/content/drive/MyDrive/MULTIVERSE_ALL_MARKET_STAGE0_PRICE_RECOVERY_v1')
receipt=OUT/'STAGE0_PRICE_BULK_RECEIPT_v1.json'
quality=OUT/'PRICE_ONLY'/'POST_BULK_PRICE_QUALITY_REPORT_v1.json'
if not receipt.exists() or not quality.exists(): raise RuntimeError('FAIL-CLOSED output receipt/quality missing')
print('
=== STAGE0 PRICE BULK RECEIPT ===')
print(receipt.read_text())
print('
=== POST-BULK QUALITY REPORT ===')
print(quality.read_text())
files.download(str(receipt))
files.download(str(quality))
